# Feedforward Neural Network (FNN)

In [ ]:
import time
import sys
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.neural_network import MLPClassifier

# ---- Feedforward Neural Network ----

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n10000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Train the model

fnn = MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), 
        max_iter=500, 
        random_state=0,
        activation='relu',
        solver='adam'
    )
    
start_time = time.process_time()
fnn.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = fnn.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
report_text = classification_report(y_test, y_pred)
num_features = fnn.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(fnn)) / 1024
param = fnn.get_params()

layer_nodes = [fnn.n_features_in_] + list(fnn.hidden_layer_sizes) + [fnn.n_outputs_]
diagram_parts = []
for i, nodes in enumerate(layer_nodes):
    layer_box = f"[{nodes}]"
    diagram_parts.append(layer_box)
    
    if i < len(layer_nodes) - 1:
        diagram_parts.append("----->")
        
horizontal_diag = " ".join(diagram_parts)

print(f"--- FNN NETWORK VISUALIZATION ---\n")
print(f"Flow: {horizontal_diag}\n\n")

print(f"Detailed Layer Breakdown:")
for i, nodes in enumerate(layer_nodes):
    label = "INPUT" if i == 0 else "OUTPUT" if i == len(layer_nodes)-1 else f"HIDDEN_{i}"
    bar = "#" * min(20, max(1, nodes // 10))
    print(f"  Layer {i} ({label:8}): {nodes:4} nodes | {bar}")

print(f"{'-'*53}")
print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")
print(f"Iter             {fnn.n_iter_}")
print(f"Loss             {round(fnn.loss_, 4)}")
print(f"Num Layers       {len(fnn.coefs_)}")

print(f"{'-'*53}")
print(f"Full Report\n{report_text}")

print(f"{'-'*53}")
print(f"Parameters:")
for p, v in param.items(): 
    print(f"{p:30}{v}")


***

## Hidden Layer Architechtuer

The architechtures of the hidden layers affect how the model thinks and performs. 

Most problems can be solved with 1-5 hidden layers, any more than this and we get into '*deep* neural nets. These deeper nerural nets have exponentially longer training times, and are not suitable to fit on consumer hardware

The most common architectures are 

**Shallow** - (64) Shallow architectures use a single hidden layer with vareying sizes.

**Funnel** - (128 > 64 > 32) Features progressively fewer neurons in each subsequent layer. 

**Constant** - (64 > 64 > 64) Maintains the exact same number of neurons across all hidden layers.

**Hourglass** - (64 > 16 > 64) Compresses the number of neurons in the middle layers before expanding them back out.

***

### Grid Search with Cross Validation

Using the `GridSearchCV` from `sklearn.model_selection`.

In [ ]:
from sklearn.model_selection import GridSearchCV

# ---- Grid Search with Cross Validation ----

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n10000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)


param_grid = {
    'hidden_layer_sizes': [
        # Shallow
        (64),
        (128),

        # Funnel
        (64, 32),
        (128, 64), 
        (256, 128, 64),
        (256, 128, 64, 32),

        # Constant / symetric
        (32, 32, 32),
        (64, 64, 64),
        (32, 32, 32, 32),
        (64, 64, 64, 64),
        
        # Hourglass / bottleneck
        (128, 64, 128),
        (64, 32, 16, 32, 64),
    ],
    'activation': ['relu', 'tanh'],   # ReLU is standard, Tanh can be better for normalized data
    'solver': ['adam'],               # Adam is generally superior for large datasets
    'alpha': [0.0001, 0.05],          # L2 penalty (Regularization)
}

grid_search = GridSearchCV(
    estimator=fnn, 
    param_grid=param_grid, 
    cv=3, 
    scoring='accuracy', 
    verbose=2, 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_fnn = grid_search.best_estimator_

y_pred = best_fnn.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = best_fnn.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(best_fnn)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {grid_search.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

***
### Random Grid Search with Cross Validaton

In [ ]:
import pandas as pd
import sys, pickle, time

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Initialize the base estimator with Early Stopping
# This is the secret to saving hours of training time
fnn = MLPClassifier(
    early_stopping=True,     # Stop if validation score doesn't improve
    validation_fraction=0.1, # Hold out 10% for internal validation
    n_iter_no_change=10,     # Patience level
    max_iter=300,            # High limit, but early stopping will likely hit first
    random_state=0
)

# Your existing architecture list
param_dist = {
    'hidden_layer_sizes': [
        # Shallow
        (64,), 
        (128,), 
        (256,),

        # Funnel
        (64, 32), 
        (128, 64, 32),
        (256, 128, 64, 32), 

        # Constant 
        (32, 32, 32), 
        (64, 64, 64), 
        (128, 128, 128),

        # Hourglass 
        (63, 32, 64),
        (128, 64, 128), 
        (64, 32, 16, 32, 64), 
        (128, 64, 32, 64, 128)
    ],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'], 
    'alpha': [0.0001, 0.001, 0.05], 
    'learning_rate_init': [0.001, 0.01, 0,1] 
}

random_search = RandomizedSearchCV(
    estimator=fnn, 
    param_distributions=param_dist, 
    n_iter=15,    # Test n random combinations
    cv=3,         # 3-fold cross validation
    scoring='accuracy', 
    verbose=2, 
    n_jobs=-1,    # Use all CPU cores
    random_state=0
)

start_wall_time = time.perf_counter()
random_search.fit(X_train, y_train)
total_wall_time = time.perf_counter() - start_wall_time

best_fnn = random_search.best_estimator_

# Final Evaluation
y_pred = best_fnn.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
size_kb = sys.getsizeof(pickle.dumps(best_fnn)) / 1024
training_time = random_search.cv_results_['mean_fit_time'][random_search.best_index_]

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {random_search.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {random_search.best_score_:.4f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
18 fits failed out of a total of 45.

Accuracy         0.9175
Precision        0.9225
Recall           0.9174
F1-Score         0.9188
Training Time    288.7232 s
Size in KB       410.1191 KB

[+] Best Parameters: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64, 64, 64), 'alpha': 0.0001, 'activation': 'tanh'}
[+] Best Cross-Validation Accuracy: 0.9167


***
### Architecture optimization using Optuna

In [ ]:
import optuna

def objective(trial):
    # Search for number of layers (1 to 5)
    n_layers = trial.suggest_int('n_layers', 1, 5)
    layers = []
    for i in range(n_layers):
        # Search for nodes in each layer (16 to 256)
        layers.append(trial.suggest_int(f'n_units_l{i}', 16, 256))
    
    clf = MLPClassifier(hidden_layer_sizes=tuple(layers), max_iter=500)
    clf.fit(X_train, y_train)
    return clf.score(X_test, y_test)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print(f"Best architecture: {study.best_params}")

***
## Training Optimization

***
### Partial Fit Training

In [4]:
import sys, pickle, time
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

csv_file = "dataset_packet_multiclass8_n1000000_shuffled.csv"
chunk_size = 50000
all_classes = np.array([0, 1, 2, 3, 4, 5, 6, 7]) 

incremental_fnn = MLPClassifier(
    solver='adam',
    learning_rate_init=0.0005, # Slightly lower for stability in incremental learning
    hidden_layer_sizes=(124, 124, 124),
    alpha=0.0001,
    activation='tanh',
    random_state=0
)

# We fit the scaler on the first chunk to establish the "language" of the data
print(f"[*] Calibrating scaler and preparing test set...")

# Load a small portion to fit the scaler and create a hold-out test set
temp_df = pd.read_csv(csv_file, nrows=100000, index_col=0)
X_temp = temp_df.drop(["label"], axis=1)
y_temp = temp_df["label"]

X_train_temp, X_test, y_train_temp, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=0)

scaler = StandardScaler()
scaler.fit(X_train_temp) 
X_test_scaled = scaler.transform(X_test) 

# Incremental training
print(f"[*] Starting incremental training on {csv_file}...")
start_time = time.perf_counter()

epochs = 3 
for epoch in range(epochs):
    print(f"[*] Epoch {epoch + 1}/{epochs}")
    
    # Re-initialize reader for each epoch
    reader = pd.read_csv(csv_file, index_col=0, chunksize=chunk_size)
    
    for i, chunk in enumerate(reader):
        X_raw = chunk.drop(["label"], axis=1)
        y_chunk = chunk["label"]
        
        # FIX: Scale the chunk using the same scaler!
        X_chunk_scaled = scaler.transform(X_raw)
        
        # Update model
        incremental_fnn.partial_fit(X_chunk_scaled, y_chunk, classes=all_classes)
        
        print(f"\r    - Processed {(i+1) * chunk_size} rows...", end="")

    # Evaluate at the end of each epoch
    y_val_pred = incremental_fnn.predict(X_test_scaled)
    epoch_acc = accuracy_score(y_test, y_val_pred)
    print(f"\n    [>] End of Epoch {epoch + 1} - Val Accuracy: {epoch_acc:.4f}")

training_time = time.perf_counter() - start_time
print(f"\n[+] Training Complete in {round(training_time, 2)} seconds")

# --- 4. FINAL EVALUATION ---
print("\n[*] Generating Final Report...")
y_pred = incremental_fnn.predict(X_test_scaled) # Use scaled test data!
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
size_kb = sys.getsizeof(pickle.dumps(incremental_fnn)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

# --- 5. SAVE MODEL ---
# with open('best_incremental_fnn.pkl', 'wb') as f:
#     pickle.dump({'model': incremental_fnn, 'scaler': scaler}, f)

[*] Calibrating scaler and preparing test set...
[*] Starting incremental training on dataset_packet_multiclass8_n1000000_shuffled.csv...
[*] Epoch 1/3
    - Processed 1000000 rows...
    [>] End of Epoch 1 - Val Accuracy: 0.8845
[*] Epoch 2/3
    - Processed 1000000 rows...
    [>] End of Epoch 2 - Val Accuracy: 0.9089
[*] Epoch 3/3
    - Processed 1000000 rows...
    [>] End of Epoch 3 - Val Accuracy: 0.9182

[+] Training Complete in 238.87 seconds

[*] Generating Final Report...
Accuracy         0.9182
Precision        0.9234
Recall           0.9182
F1-Score         0.9193
Training Time    238.8658 s
Size in KB       1506.3262 KB
